# 6.31 — Neural Architecture Search

NAS treats architecture as the object being optimized, balancing validation quality against search cost.

NAS optimizes the architecture choice $a$, not just weights $w_a$. The lesson pitfall is selection by training loss: the notebook searches small CPU-safe networks and reports validation accuracy against number of tried configurations.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(42)
random.seed(42)

def clf_digits_ladder():
    """A harder image-as-tabular classification ladder for DL topics (part 6).

    D1 XOR -> D2 blobs -> D3 noisy moons -> D4 sklearn digits (10-class, 64-D) ->
    D5 digits with label noise + feature noise (distribution shift).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def split_scale(X, y):
    stratify = y if min(np.bincount(y)) >= 2 else None
    x_tr, x_te, y_tr, y_te = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_softmax_linear(x_tr, y_tr, x_te, epochs=220, lr=0.25, mask=None, theta0=None, quant_bits=None):
    classes = np.unique(y_tr)
    n_classes = int(classes.max()) + 1
    rng = np.random.default_rng(7)
    if theta0 is None:
        W = rng.normal(0.0, 0.05, size=(x_tr.shape[1], n_classes))
        b = np.zeros(n_classes)
    else:
        W = theta0[0].copy()
        b = theta0[1].copy()
    if mask is None:
        mask = np.ones_like(W)
    Y = one_hot(y_tr, n_classes)
    for epoch in range(epochs):
        logits = x_tr @ (W * mask) + b
        probs = softmax(logits)
        grad_logits = (probs - Y) / len(y_tr)
        grad_W = x_tr.T @ grad_logits
        grad_b = grad_logits.sum(axis=0)
        if quant_bits is not None:
            grad_W = quantize_fixed(grad_W, quant_bits)
            grad_b = quantize_fixed(grad_b, quant_bits)
        W = W - lr * grad_W * mask
        b = b - lr * grad_b
    scores = x_te @ (W * mask) + b
    return scores.argmax(axis=1), (W, b)


def quantize_fixed(values, bits):
    values = np.asarray(values, dtype=float)
    levels = 2 ** bits - 1
    clipped = np.clip(values, -2.0, 2.0)
    scaled = np.round((clipped + 2.0) * levels / 4.0)
    return scaled * 4.0 / levels - 2.0


def mlp_predict(x_tr, y_tr, x_te, hidden=(16,), alpha=0.0001, max_iter=260):
    clf = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.02,
        max_iter=max_iter,
        random_state=3,
    )
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def ladder_preview(rungs):
    rows = []
    for name, X, y in rungs:
        rows.append((name, X.shape, int(len(np.unique(y)))))
    for name, shape, classes in rows:
        print(f"{name:34s} shape={shape} classes={classes}")
    print("D1 sample X:")
    print(rungs[0][1])
    print("D1 labels:")
    print(rungs[0][2])


def evaluate_accuracy_ladder(method):
    rows = []
    rungs = clf_digits_ladder()
    for name, X, y in rungs:
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        preds, artifact = method(x_tr, y_tr, x_te, name)
        acc = accuracy_score(y_te, preds)
        rows.append({"name": name, "metric": acc, "artifact": artifact, "X": X, "y": y})
    for row in rows:
        print(f"{row['name']:34s} accuracy={row['metric']:.3f}")
    return rows


def plot_results(rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, row in zip(axes[0], rows):
        X = row["X"]
        y = row["y"]
        if X.shape[1] > 2:
            pts = PCA(n_components=2, random_state=0).fit_transform(X)
        else:
            pts = X
        ax.scatter(pts[:, 0], pts[:, 1], c=y, s=12, cmap="tab10", alpha=0.75)
        ax.set_title(row["name"].split(" (")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    xs = np.arange(1, len(rows) + 1)
    ys = [row["metric"] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel(ylabel)
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## The concept, built once (D1)
$$a^*=\arg\min_{a\in\mathcal{A}} L_{val}(w_a,a)$$

On D1 we compare three tiny architectures by validation loss and assert the argmin. The real ladder reuses the same search idea on increasingly hard datasets.

In [ ]:
def architecture_search(loss_table):
    best = min(loss_table, key=lambda row: row["val_loss"])
    return best


d1_space = [
    {"name": "linear", "train_loss": 0.42, "val_loss": 0.69, "configs": 1},
    {"name": "wide_8", "train_loss": 0.05, "val_loss": 0.22, "configs": 2},
    {"name": "deep_4_4", "train_loss": 0.01, "val_loss": 0.31, "configs": 3},
]
best_demo = architecture_search(d1_space)
assert best_demo["name"] == "wide_8"
assert round(best_demo["val_loss"], 2) == 0.22
print("D1 selected architecture:", best_demo)

The assertion above pins the notebook to the lesson's worked numbers before we scale the same idea up the ladder.

In [ ]:
print('D1 concept verified for 6.31')

## The dataset ladder
We use the shared F5 `clf_digits_ladder()` exactly: XOR, blobs, noisy moons, real digits, then noisy shifted digits.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1-D5

In [ ]:
def search_small_nets(x_tr, y_tr, x_te, y_te):
    candidates = [
        ("linear", ()),
        ("wide_8", (8,)),
        ("wide_16", (16,)),
        ("deep_16_8", (16, 8)),
    ]
    x_fit, x_val, y_fit, y_val = train_test_split(
        x_tr,
        y_tr,
        test_size=0.35,
        random_state=1,
        stratify=y_tr if min(np.bincount(y_tr)) >= 2 else None,
    )
    records = []
    for idx, (name, hidden) in enumerate(candidates, start=1):
        if hidden:
            clf = MLPClassifier(
                hidden_layer_sizes=hidden,
                max_iter=220,
                learning_rate_init=0.02,
                random_state=idx,
            )
        else:
            clf = LogisticRegression(max_iter=1000)
        clf.fit(x_fit, y_fit)
        val_probs = clf.predict_proba(x_val)
        val_loss = log_loss(y_val, val_probs, labels=np.unique(y_tr))
        records.append({"name": name, "val_loss": val_loss, "model": clf, "configs": idx})
    best = min(records, key=lambda row: row["val_loss"])
    preds = best["model"].predict(x_te)
    return preds, {"best": best["name"], "records": records}


def nas_method(x_tr, y_tr, x_te, name):
    preds, artifact = search_small_nets(x_tr, y_tr, x_te, y_tr)
    return preds, artifact


rows = evaluate_accuracy_ladder(nas_method)
for row in rows:
    print(row["name"], "best=", row["artifact"]["best"])

## Results visualization
Top row: rung artifacts in two dimensions. Bottom-left: the one tracked metric from D1 to D5.

In [ ]:
plot_results(rows, 'NAS accuracy versus rung after four configs')

## Pitfall on the hardest rung
Pitfall on D5: choosing by training loss overfits the search. The fix is to select by validation loss and report the number of tried configurations.

In [ ]:
name, X5, y5 = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X5, y5)
preds, artifact = search_small_nets(x_tr, y_tr, x_te, y_te)
for record in artifact["records"]:
    model = record["model"]
    train_loss = log_loss(y_tr, model.predict_proba(x_tr), labels=np.unique(y_tr))
    print(f"{record['name']:10s} train_loss={train_loss:.3f} val_loss={record['val_loss']:.3f}")
print("validation-selected best:", artifact["best"])
print("search configs tried:", len(artifact["records"]))

## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline such as majority-class accuracy or untrained random predictions.
- Cheap sanity check: rerun with the same seed and confirm the D1 arithmetic assertions still pass.
- Ablation: turn off the key idea (generated context, routing, spikes, validation search, reset, capacity sweep, or loss scaling) and expect the hardest-rung metric to worsen or become less reliable.
- Failure signals: unstable curves, shape mismatches, nearly constant predictions, or a D5 result that improves only by using training labels for selection.

Practice prompts:
1. Change one hyperparameter in the pitfall cell and explain whether the metric moved for the reason the lesson predicts.

In [ ]:
# Try it here.

2. Replace D5 with a smaller subset and predict which failure signal becomes easier or harder to see.

In [ ]:
# Try it here.

3. Add one baseline row to the summary curve and decide whether the specialized method earned its complexity.

In [ ]:
# Try it here.